# Tiny Recursive Reasoning Model (TRM) Workspace

This notebook is optimized for training and evaluating TRM models from scratch in the Modal environment.

In [ ]:
# ── 1. Environment and Path Setup ───────────────────────────────────────────
import os
import sys
from pathlib import Path

os.chdir("/root/EdgeTRM")
print("Working Directory:", os.getcwd())

# Add TinyRecursiveModels to system path
repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))
print("trm_root added to sys.path:", trm_root)

In [ ]:
# ── 2. Fix Duplicate Modules on Modal ────────────────────────────────────────
# Replaces duplicate trm.py with a symlink to prevent dual-import namespace conflicts
modal_top = "/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/trm.py"
modal_sub = "/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels/models/recursive_reasoning/trm.py"

if os.path.exists(modal_sub) and os.path.exists(modal_top) and not os.path.islink(modal_top):
    os.rename(modal_top, modal_top + ".bak")
    os.symlink(modal_sub, modal_top)
    print("✓ Successfully symlinked trm.py files to resolve namespace conflicts!")
else:
    print("✓ Symlink already exists or paths are aligned.")

In [ ]:
# ── 3. High-Performance Per-Puzzle Evaluator ─────────────────────────────────
# Implements true paper-aligned search, cache transformations, and puzzle batching
import torch
import numpy as np
import json
import os
import time
from tqdm.notebook import tqdm

def get_inner(mdl):
    if hasattr(mdl, "model"):
        return get_inner(mdl.model)
    return mdl

@torch.no_grad()
def evaluate_arc_per_puzzle(mdl, loader, device="cpu", n_sup_max=16, max_batches=None, return_pass2=False):
    from models.recursive_reasoning.trm import (
        TinyRecursiveReasoningModel_ACTV1Carry,
        TinyRecursiveReasoningModel_ACTV1InnerCarry,
    )
    from dataset.build_arc_dataset import inverse_aug, grid_hash, arc_grid_to_np, PuzzleIdSeparator
    from evaluators.arc import _crop
    import json
    import os
    import numpy as np
    import time
    from tqdm.notebook import tqdm

    inner = get_inner(mdl)
    inner.eval()
    inner = inner.to(device)

    # Load mappings
    with open(os.path.join(DATA_DIR, "identifiers.json"), "r") as f:
        identifier_map = json.load(f)
    with open(os.path.join(DATA_DIR, "test_puzzles.json"), "r") as f:
        test_puzzles = json.load(f)

    # High-performance caching of inverse_aug
    aug_cache = {}
    def get_aug(pid):
        if pid not in aug_cache:
            name = identifier_map[pid]
            aug_cache[pid] = inverse_aug(name)
        return aug_cache[pid]

    # High-performance caching of _crop to eliminate Numba overhead on repetitive sequences
    crop_cache = {}
    def get_crop(seq):
        seq_bytes = seq.tobytes()
        if seq_bytes not in crop_cache:
            crop_cache[seq_bytes] = _crop(seq)
        return crop_cache[seq_bytes]

    # Precompute canonical input hashes
    precomputed_input_info = {}
    ds = loader.dataset
    if hasattr(ds, "inputs") and hasattr(ds, "per_sample_pids"):
        try:
            puzzle_ids_arr = None
            puzzle_ptr_arr = None
            for split in ["test", "train"]:
                ids_path = os.path.join(DATA_DIR, split, "all__puzzle_identifiers.npy")
                ptr_path = os.path.join(DATA_DIR, split, "all__puzzle_indices.npy")
                if os.path.exists(ids_path):
                    ptr = np.load(ptr_path)
                    if ptr[-1] == len(ds):
                        puzzle_ids_arr = np.load(ids_path)
                        puzzle_ptr_arr = ptr
                        break
            if puzzle_ids_arr is not None and puzzle_ptr_arr is not None:
                puzzle_test_hashes = {
                    name: [grid_hash(arc_grid_to_np(pair["input"])) for pair in pz["test"]]
                    for name, pz in test_puzzles.items()
                }
                for j in range(len(puzzle_ids_arr)):
                    pid = int(puzzle_ids_arr[j])
                    if pid == 0: continue
                    name = identifier_map[pid]
                    orig_name = name.split(PuzzleIdSeparator)[0]
                    if orig_name not in test_puzzles: continue
                    E = len(test_puzzles[orig_name]["test"])
                    start_ptr = int(puzzle_ptr_arr[j])
                    end_ptr = int(puzzle_ptr_arr[j+1])
                    for k in range(start_ptr, end_ptr):
                        test_example_index = (k - start_ptr) % E
                        input_hash = puzzle_test_hashes[orig_name][test_example_index]
                        precomputed_input_info[k] = (orig_name, input_hash)
        except Exception as e:
            print(f"[WARN] Input hash precomputation failed, falling back: {e}")

    local_hmap = {}       # pred_hash -> canonical grid np.ndarray
    local_preds = {}      # orig_name -> {input_hash -> [(pred_hash, q_val), ...]}

    t0 = time.time()
    
    # Bypass DataLoader collation completely to eliminate sequential __getitem__ CPU tensor allocation overhead
    ds = loader.dataset
    inputs_np = ds.inputs
    labels_np = ds.labels
    pids_np = ds.per_sample_pids
    num_samples = len(inputs_np)
    batch_size = loader.batch_size if hasattr(loader, "batch_size") else 512
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    pbar = tqdm(range(num_batches), desc="Evaluating per-puzzle batches", leave=False)
    for batch_idx in pbar:
        if max_batches is not None and batch_idx >= max_batches:
            break

        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_samples)
        
        x_batch = torch.from_numpy(inputs_np[start_idx:end_idx]).to(device, dtype=torch.long)
        y_true  = torch.from_numpy(labels_np[start_idx:end_idx]).to(device, dtype=torch.long)
        pids    = torch.from_numpy(pids_np[start_idx:end_idx]).to(device, dtype=torch.long)

        batch = {
            "inputs":             x_batch.to(torch.int32),
            "labels":             y_true.to(torch.int32),
            "puzzle_identifiers": pids.to(torch.int32),
        }

        carry = inner.initial_carry(batch)
        ic    = carry.inner_carry
        cast  = lambda t: t.to(device)
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )

        last_outputs = None
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            last_outputs = outputs
            if carry.halted.all():
                break

        if last_outputs is None:
            continue

        preds_batch = last_outputs["logits"].argmax(-1).cpu().numpy()  # (B, seq_len)
        q_logits    = last_outputs.get("q_halt_logits", torch.zeros(preds_batch.shape[0], device=device))
        q_values    = q_logits.sigmoid().cpu().numpy().flatten()    # (B,)

        inputs_cpu  = inputs_np[start_idx:end_idx]
        pids_cpu    = pids_np[start_idx:end_idx]

        for i in range(preds_batch.shape[0]):
            identifier = pids_cpu[i]
            if identifier == 0:  # Skip blank padding
                continue

            orig_name, _inverse_fn = get_aug(identifier)
            pred_seq = preds_batch[i]
            q_val = float(q_values[i])

            sample_idx = start_idx + i
            if sample_idx in precomputed_input_info and precomputed_input_info[sample_idx][0] == orig_name:
                input_hash = precomputed_input_info[sample_idx][1]
            else:
                inp_seq = inputs_cpu[i]
                input_grid = _inverse_fn(get_crop(inp_seq))
                input_hash = grid_hash(input_grid)

            # Crop and inverse transform prediction
            pred_grid = _inverse_fn(get_crop(pred_seq))
            pred_hash = grid_hash(pred_grid)

            local_hmap[pred_hash] = pred_grid

            local_preds.setdefault(orig_name, {})
            local_preds[orig_name].setdefault(input_hash, [])
            local_preds[orig_name][input_hash].append((pred_hash, q_val))

    # Compute final accuracies
    n_puzzles = 0
    correct = [0, 0]
    cell_hits = 0
    n_cells = 0

    evaluated_puzzles = [name for name in test_puzzles.keys() if name in local_preds]
    for name in evaluated_puzzles:
        puzzle = test_puzzles[name]
        n_puzzles += 1
        num_correct = [0, 0]
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            label_hash = grid_hash(out_grid)

            p_map = {}
            for h, q in local_preds[name].get(input_hash, []):
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q

            if not len(p_map):
                continue

            # Pass@1: pure frequency voting
            p_map_freq = sorted(p_map.items(), key=lambda kv: kv[1][0], reverse=True)
            if p_map_freq[0][0] == label_hash:
                num_correct[0] = 1

            # Pass@2: average Q-value voting
            for h, stats in p_map.items():
                stats[1] /= stats[0]
            p_map_q = sorted(p_map.items(), key=lambda kv: kv[1][1], reverse=True)
            if p_map_q[0][0] == label_hash:
                num_correct[1] = 1

        correct[0] += num_correct[0]
        correct[1] += num_correct[1]

        # Cell Accuracy
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)

            preds_list = local_preds.get(name, {}).get(input_hash, [])
            if not preds_list:
                continue

            p_map = {}
            for h, q in preds_list:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            for h, stats in p_map.items():
                stats[1] /= stats[0]

            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
            top_hash = p_map_sorted[0][0]
            top_grid = local_hmap[top_hash]

            if top_grid.shape == out_grid.shape:
                cell_hits += (top_grid == out_grid).sum()
                n_cells += out_grid.size
            else:
                n_cells += out_grid.size

    cell_acc = cell_hits / n_cells if n_cells > 0 else 0.0
    pass_1_acc = correct[0] / n_puzzles if n_puzzles > 0 else 0.0
    pass_2_acc = correct[1] / n_puzzles if n_puzzles > 0 else 0.0
    elapsed = time.time() - t0
    ms_per_puzzle = elapsed / n_puzzles if n_puzzles > 0 else 0.0

    if return_pass2:
        return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle * 1000, n_puzzles
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle * 1000, n_puzzles

---
## Section 5 — Training a TRM Model from Scratch

In this section, we implement a premium-grade training pipeline to train a fresh `TinyRecursiveReasoningModel_ACTV1` model from scratch on the ARC-AGI dataset.

### Training Details & Hyperparameters
- **Main Optimizer**: `AdamAtan2` (Atan2-based gradient updates for superior learning) with fallback to `AdamW` if not installed.
- **Embedding Optimizer**: `CastedSparseEmbeddingSignSGD_Distributed` to update sparse puzzle embeddings using SignSGD.
- **Learning Rate Schedule**: Cosine learning rate decay with a linear warmup phase.
- **Model Configuration**: Paper-aligned architecture ($H_{cycles}=4, L_{cycles}=4, L_{layers}=2, hidden\_size=512$, vocab_size=12, seq_len=900, puzzle embedding dimension 512, halt steps 16).
- **VRAM Management**: Batch size of `256` prevents GPU memory swapping and PCIe bottlenecking, ensuring maximum local execution speed.

In [ ]:
# ── 5.1  Model & Dataset Initialization ──────────────────────────────────────
import os
import sys
import math
import copy
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np
from tqdm.notebook import tqdm

# Add repo to path if needed
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
from models.losses import ACTLossHead
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed

# Attempt to load AdamAtan2 from paper, fallback to AdamW
try:
    from adam_atan2_pytorch import AdamAtan2
    print("Successfully imported AdamAtan2!")
except ModuleNotFoundError:
    from torch.optim import AdamW as AdamAtan2
    print("WARNING: adam_atan2_pytorch not found, using AdamW as fallback.")

# Hyperparameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "./data1/arc2test-aug-1000"
CHECKPOINT_DIR = "./checkpoints/trm_scratch"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Optimizing for NVIDIA H200 (141GB VRAM) ───────────────────────────────────
BATCH_SIZE = 1536  # Fully saturates Hopper Tensor Cores without swapping
# BATCH_SIZE = 256  # VRAM-friendly batch size to prevent swapping/PCIe bottlenecks
LR = 4e-4          # Scaled learning rate for larger batch size
PUZZLE_EMB_LR = 4e-2
WEIGHT_DECAY = 0.1
PUZZLE_EMB_WEIGHT_DECAY = 0.1
TOTAL_STEPS = 5000
WARMUP_STEPS = 500
LR_MIN_RATIO = 0.1
CHECKPOINT_INTERVAL = 500
NUM_PUZZLE_IDENTIFIERS = 225972

NUM_PUZZLE_IDENTIFIERS = 225972  # Vocabulary size matching the dataset size (including <blank>)

print(f"Device: {DEVICE}")
print(f"Initializing loaders from: {DATA_DIR}")

# 1. Build Datasets & Loaders
train_ds_config = PuzzleDatasetConfig(
    seed=0,
    dataset_paths=[DATA_DIR],
    global_batch_size=BATCH_SIZE,
    test_set_mode=False,
    epochs_per_iter=1,
    rank=0,
    num_replicas=1
)
train_ds = PuzzleDataset(train_ds_config, split="train")

test_ds_config = PuzzleDatasetConfig(
    seed=0,
    dataset_paths=[DATA_DIR],
    global_batch_size=BATCH_SIZE,
    test_set_mode=True,
    epochs_per_iter=1,
    rank=0,
    num_replicas=1
)
test_ds = PuzzleDataset(test_ds_config, split="test")

train_loader = DataLoader(train_ds, batch_size=None, num_workers=1, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=None, num_workers=1, pin_memory=True)

print("Loading dataset metadata...")
metadata = train_ds.metadata
print(f"Vocab Size: {metadata.vocab_size}, Sequence Length: {metadata.seq_len}")
print(f"Total groups: {metadata.total_groups}, Total puzzles: {metadata.total_puzzles}")

# 2. Instantiate TRM Model with Loss Head
model_config = {
    "batch_size": BATCH_SIZE,
    "seq_len": metadata.seq_len,
    "puzzle_emb_ndim": 512,
    "num_puzzle_identifiers": NUM_PUZZLE_IDENTIFIERS,
    "vocab_size": metadata.vocab_size,
    "H_cycles": 4,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4.0,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

print("Initializing fresh TinyRecursiveReasoningModel_ACTV1 model...")
base_model = TinyRecursiveReasoningModel_ACTV1(model_config)
model = ACTLossHead(base_model, loss_type="stablemax_cross_entropy")
model = model.to(DEVICE)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# 3. Instantiate Optimizers
# Embedding SignSGD optimizer
emb_optimizer = CastedSparseEmbeddingSignSGD_Distributed(
    model.model.puzzle_emb.buffers(),
    world_size=1,
    lr=PUZZLE_EMB_LR,
    weight_decay=PUZZLE_EMB_WEIGHT_DECAY
)

# Model AdamAtan2 / AdamW optimizer
main_optimizer = AdamAtan2(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95)
)

print("Model, datasets, and optimizers successfully initialized!")

RuntimeError: Only a single TORCH_LIBRARY can be used to register the namespace triton; please put all of your definitions in a single TORCH_LIBRARY block.  If you were trying to specify implementations, consider using TORCH_LIBRARY_IMPL (which can be duplicated).  If you really intended to define operators for a single namespace in a distributed way, you can use TORCH_LIBRARY_FRAGMENT to explicitly indicate this.  Previous registration of TORCH_LIBRARY was registered at /dev/null:2630; latest registration was registered at /dev/null:2630

In [28]:
# ── 5.2  Training & Validation Loop ──────────────────────────────────────────
def get_lr_factor(step, total_steps, warmup_steps, min_ratio):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return min_ratio + max(0.0, (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress)))

print(f"Starting training from scratch for {TOTAL_STEPS} steps...")
model.train()

train_iter = iter(train_loader)
running_loss = 0.0
running_acc = 0.0
running_em = 0.0
running_steps = 0.0
log_window = 100

pbar = tqdm(range(1, TOTAL_STEPS + 1), desc="Training steps")
for step in pbar:
    try:
        set_name, batch, eff_batch_size = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        set_name, batch, eff_batch_size = next(train_iter)
        
    # Scale learning rates according to scheduler
    lr_scale = get_lr_factor(step, TOTAL_STEPS, WARMUP_STEPS, LR_MIN_RATIO)
    for param_group in main_optimizer.param_groups:
        param_group['lr'] = LR * lr_scale
    for param_group in emb_optimizer.param_groups:
        param_group['lr'] = PUZZLE_EMB_LR * lr_scale
        
    # Move batch to device
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    
    # Forward pass
    carry = model.initial_carry(batch)
    carry, loss, metrics, _, _ = model(carry=carry, batch=batch, return_keys=[])
    
    # Backward pass & Optimize
    loss_normalized = loss / BATCH_SIZE
    loss_normalized.backward()
    
    # Apply steps & Zero grads
    main_optimizer.step()
    main_optimizer.zero_grad()
    
    emb_optimizer.step()
    emb_optimizer.zero_grad()
    
    # Update metrics
    count = max(float(metrics.get("count", BATCH_SIZE)), 1.0)
    step_loss = float(loss.item()) / BATCH_SIZE
    step_acc = float(metrics.get("accuracy", 0.0)) / count
    step_em = float(metrics.get("exact_accuracy", 0.0)) / count
    step_steps = float(metrics.get("steps", 0.0)) / count
    
    running_loss += (step_loss - running_loss) / min(step, log_window)
    running_acc += (step_acc - running_acc) / min(step, log_window)
    running_em += (step_em - running_em) / min(step, log_window)
    running_steps += (step_steps - running_steps) / min(step, log_window)
    
    pbar.set_postfix({
        "loss": f"{running_loss:.4f}",
        "acc": f"{running_acc*100:.2f}%",
        "em": f"{running_em*100:.2f}%",
        "steps": f"{running_steps:.1f}"
    })
    
    # Periodic evaluation and checkpointing
    if step % CHECKPOINT_INTERVAL == 0:
        # Save checkpoint
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"step_{step}.pt")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"\n[Step {step}] Checkpoint saved to: {checkpoint_path}")
        
        # Run fast validation evaluation
        print(f"[Step {step}] Running fast per-puzzle evaluation...")
        p1, cell, ms, npuzz = evaluate_arc_per_puzzle(
            model, test_loader, device=DEVICE, n_sup_max=16, return_pass2=False
        )
        print(f"Validation Results -> Pass@1: {p1*100:.2f}% | Cell Acc: {cell*100:.2f}% | Latency: {ms:.2f} ms/puzzle")
        model.train()  # Restore training mode

NameError: name 'TOTAL_STEPS' is not defined